In [ ]:
# storing the decent message here
clean_msg= []
seen_users =[]
seen_date=[]
msg_count = 0
noise_count = 0
media_junk = 0
deleted_msg = 0
sys_msg = 0
people_count =0
day_count = 0
media_count = 0
deleted_msg = 0
msg_part = 0
seen_dates = []
days_count = 0
sender_name =0
user_split = 0
time_parts = ''
sender =''
data_str = 0

clean_data = []
# list
seen_users = []
seen_dates = []

# running the file
with open('/content/hostel_bois.txt', 'r', encoding='utf-8') as f:
    for raw in f:
        txt = raw.strip()

        # blank
        if not txt:
            continue

        # removing time stapms
        if ' - ' in txt:
            parts = txt.split(' - ', 1)
            time_part = parts[0]
            msg_part = parts[1]

            # splitting sender name
            if ': ' in msg_part:
                user_split = msg_part.split(': ', 1)
                sender = user_split[0]
                actual_text = user_split[1].strip()

                # removing non essentials
                if "<Media omitted>" in actual_text:
                    media_count += 1
                elif "This message was deleted" in actual_text:
                    deleted_count += 1
                else:
                    # san=ving meaningfull message
                    clean_data.append({
                        'time': time_part,
                        'name': sender,
                        'message': actual_text
                    })
                    msg_count += 1

                    # checking people
                    if sender not in seen_users:
                        seen_users.append(sender)
                        people_count += 1

                    # checking dates and tmie of messages
                    if ',' in time_part:
                        date_str = time_part.split(',')[0]
                        if date_str not in seen_dates:
                            seen_dates.append(date_str)
                            days_count += 1
            else:
              sys_count += 1


print(f"Successfully parsed {msg_count} messages from {people_count} participants over {days_count} days, skipped {sys_count} system messages, {media_count} media-omitted, {deleted_count} deleted messages")

Successfully parsed 3127 messages from 6 participants over 60 days, skipped 16 system messages, 32 media-omitted, 60 deleted messages


In [ ]:
# list we used
unique_users = []
person_counts = {}

total_msgs = 0
total_participants = 0
first_msg_date = ""
last_msg_date = ""

for msg in clean_data:
    sender = msg['name']
    timestamp = msg['time']

    total_msgs += 1

    if sender not in unique_users:
        unique_users.append(sender)
        total_participants += 1

    # each person directory
    if sender in person_counts:
        person_counts[sender] += 1
    else:
        person_counts[sender] = 1

    # splitting the dates
    if ',' in timestamp:
        parts = timestamp.split(',')
        date_val = parts[0]
        if first_msg_date == "":
            first_msg_date = date_val
        last_msg_date = date_val


# list we used
ranking_list = []
for user in unique_users:
    count = person_counts[user]
    ranking_list.append([user, count])


for i in range(total_participants):
    for j in range(0, total_participants - i - 1):
        if ranking_list[j][1] < ranking_list[j+1][1]:
            temp = ranking_list[j]
            ranking_list[j] = ranking_list[j+1]
            ranking_list[j+1] = temp

# f-sstring we used
print("  group overview ")
print(f"Total Messages: {total_msgs}")
print(f"Date Range: {first_msg_date} to {last_msg_date}")
print(f"Total Participants: {total_participants}")
print("\n  message per person  ")

# obseravtion
for item in ranking_list:
    user_name = item[0]
    user_total = item[1]
    print(f"{user_name}: {user_total} messages")

  group overview 
Total Messages: 3127
Date Range: 01/04/24 to 30/05/24
Total Participants: 6

  message per person  
Rahul: 940 messages
Priya: 712 messages
Neha: 624 messages
Aman: 484 messages
Karan: 345 messages
Vikas: 22 messages


In [ ]:
# making dictionary
day_totals = {}
hour_totals = {}

for item in clean_data:
    time_str = item['time']

    # getting the day and hour
    if ',' in time_str:
        time_parts = time_str.split(',')
        date_only = time_parts[0]

        # writing the time
        time_of_day = time_parts[1].strip()

        if ':' in time_of_day:
            # making it in digital system
            hour_only = time_of_day.split(':')[0]

            # wrting the day
            if date_only in day_totals:
                day_totals[date_only] += 1
            else:
                day_totals[date_only] = 1

            # writing the hours
            if hour_only in hour_totals:
                hour_totals[hour_only] += 1
            else:
                hour_totals[hour_only] = 1

# finnding the max
peak_day = max(day_totals, key=day_totals.get)
peak_day_amount = day_totals[peak_day]

peak_hour = max(hour_totals, key=hour_totals.get)
peak_hour_amount = hour_totals[peak_hour]

# results
print("   absolute peaks   ")
print(f"Busiest Day: {peak_day} with {peak_day_amount} messages")
print(f"Busiest Hour: {peak_hour}:00 with {peak_hour_amount} messages")

   absolute peaks   
Busiest Day: 04/05/24 with 74 messages
Busiest Hour: 18:00 with 244 messages


In [ ]:
import numpy as np

# 1. each member track
user_to_row = {}
current_row_idx = 0

for item in clean_data:
    sender = item['name']
    if sender not in user_to_row:
        user_to_row[sender] = current_row_idx
        current_row_idx += 1

# 2.fromation of matrix
heatmap_matrix = np.zeros((current_row_idx, 24), dtype=int)

# 3. for messaages
for msg in clean_data:
    sender = msg['name']
    time_str = msg['time']

    if ',' in time_str:
        time_parts = time_str.split(',')
        time_of_day = time_parts[1].strip()

        if ':' in time_of_day:
            hour_str = time_of_day.split(':')[0]
            hour_int = int(hour_str)
            person_row = user_to_row[sender]
            heatmap_matrix[person_row, hour_int] += 1

print("     activity heatmap   ")
print("Hours: 00 01 02 03 04 05 06 07 08 09 10 11 12 13 14 15 16 17 18 19 20 21 22 23")

matrix_row = 0
for name in user_to_row:
    person_hours = heatmap_matrix[matrix_row, :]
    person_max = np.max(person_hours)

    # for name
    short_name = name[:6]
    print(f"{short_name} |", end="")

    for hour_count in person_hours:
        if person_max == 0 or hour_count == 0:
            print("  ", end="")
        else:
            ratio = hour_count / person_max

            if ratio <= 0.25:
                print(". ", end="")
            elif ratio <= 0.50:
                print("--", end="")
            elif ratio <= 0.75:
                print("\\", end="")
            else:
                print("**", end="")

    # result
    print()
    matrix_row += 1

     activity heatmap   
Hours: 00 01 02 03 04 05 06 07 08 09 10 11 12 13 14 15 16 17 18 19 20 21 22 23
Rahul |. . . . . . . . . . . . \----\\--**\--**\\
Priya |            . --\********\\----\\**\----. 
Karan |              . ----\--**\**\\\\**\--. . 
Neha |          --. . \****--\\--. \******\------
Aman |\****\**                  . . . . . . . .   \
Vikas |              --\----  --\  ----**\\------\


In [ ]:
# list of words to skip
stop_words = ['i', 'is', 'the', 'a', 'and', 'or', 'to', 'of', 'in', 'on', 'for', 'you', 'my', 'it', 'that', 'this', 'me']

word_counts = {}

# punctutaion
punct = '.,!?"\'()-:;'

# loop used
for msg in clean_data:
    actual_text = msg['message']

    words = actual_text.split()

    for w in words:
        clean_word = w.lower()

        clean_word = clean_word.strip(punct)

        if clean_word == "" or clean_word in stop_words:
            continue

        if clean_word in word_counts:
            word_counts[clean_word] += 1
        else:
            word_counts[clean_word] = 1

# building lift
ranking_list = []
for word in word_counts:
    count = word_counts[word]
    ranking_list.append([count, word])

top_words = sorted(ranking_list, reverse=True)

max_word_count = top_words[0][0]

print("    TOP 20 GROUP WORDS   ")

# printing of top 20
items_printed = 0

for item in top_words:
    if items_printed == 20:
        break

    word_total = item[0]
    word_name = item[1]

    print(f"{word_name:10} | ", end="")

    # the lenght of block
    num_blocks = int((word_total / max_word_count) * 20)

    # for horizontal bars
    block_count = 0
    while block_count < num_blocks:
        print("***", end="")
        block_count += 1

    # result
    print(f" {word_total}")
    items_printed += 1

    TOP 20 GROUP WORDS   
was        | ************************************************************ 385
how        | ************************************************ 321
guys       | ************************************************ 318
so         | ********************************************* 292
about      | ****************************************** 274
hai        | *************************************** 268
am         | *************************************** 260
today      | *************************************** 257
at         | *************************************** 257
he         | ********************************* 220
his        | ********************************* 217
have       | ****************************** 209
just       | ****************************** 208
which      | ****************************** 202
everyone   | *************************** 187
telling    | *************************** 179
from       | *************************** 174
up         | *******************

In [ ]:
from datetime import datetime

# response time
response_sec = {}
response_cnt = {}


user_streaks = {}
user_last_dt = {}
user_last_str = {}

prev_sender = ""
prev_dt = ""
chat_start_dt = ""
chat_start_str = ""

for msg in clean_data:
    sender = msg['name']

    # checking the starting
    if chat_start_dt == "":
        chat_start_dt = datetime.strptime(msg['time'], '%d/%m/%y, %H:%M')
        chat_start_str = msg['time']

    if sender not in user_streaks:
        user_streaks[sender] = [0, "", ""]
        response_sec[sender] = 0.0
        response_cnt[sender] = 0

# Loop 2: The actual logic
for msg in clean_data:
    sender = msg['name']
    current_time_str = msg['time']
    current_dt = datetime.strptime(current_time_str, '%d/%m/%y, %H:%M')


    #  Average Response Time
    if prev_sender != "" and prev_sender != sender:

        gap = (current_dt - prev_dt).total_seconds()
        response_sec[sender] += gap
        response_cnt[sender] += 1

    prev_sender = sender
    prev_dt = current_dt

    # finding silent streaks
    if sender in user_last_dt:
        time_diff = current_dt - user_last_dt[sender]
        streak_days = time_diff.days - 1
        start_ref_str = user_last_str[sender]
    else:
        # checking time differnce
        time_diff = current_dt - chat_start_dt
        streak_days = time_diff.days - 1
        start_ref_str = chat_start_str

    # neglecting negative streak days
    if streak_days < 0:
        streak_days = 0

    # streakes maintained by people
    if streak_days > user_streaks[sender][0]:
        user_streaks[sender] = [streak_days, start_ref_str, current_time_str]

    # seeing last messages
    user_last_dt[sender] = current_dt
    user_last_str[sender] = current_time_str

# finding fast and slow users
fast_user = ""
fast_val = 999999999.0
slow_user = ""
slow_val = 0.0

for sender in response_sec:
    if response_cnt[sender] > 0:
        avg = response_sec[sender] / response_cnt[sender]
        if avg < fast_val:
            fast_val = avg
            fast_user = sender
        if avg > slow_val:
            slow_val = avg
            slow_user = sender

# time conversion
if fast_val > 3600:
    f_time = fast_val / 3600
    f_label = "hours"
else:
    f_time = fast_val / 60
    f_label = "minutes"

if slow_val > 3600:
    s_time = slow_val / 3600
    s_label = "hours"
else:
    s_time = slow_val / 60
    s_label = "minutes"

streak_list = []
u_count = 0
for sender in user_streaks:
    streak_list.append([sender, user_streaks[sender]])
    u_count += 1

for i in range(u_count):
    for j in range(0, u_count - i - 1):
        if streak_list[j][1][0] < streak_list[j+1][1][0]:
            temp = streak_list[j]
            streak_list[j] = streak_list[j+1]
            streak_list[j+1] = temp


print("     response   pattern ")
print(f"Fastest replier : {fast_user} (avg {f_time:.1f} {f_label})")
print(f"Slowest replier : {slow_user} (avg {s_time:.1f} {s_label})")
print("\nLONGEST SILENT STREAKS (consecutive days with zero messages)")

# tracking the months
month_names = ["", "Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

for item in streak_list:
    name = item[0]
    days = item[1][0]
    s_str = item[1][1]
    e_str = item[1][2]

    if days == 0:
        print(f"{name} : 0 days (never went silent)")
    else:
        # slicing the string
        s_date = s_str.split(',')[0].replace('-', '/').split('/')
        s_day = s_date[0]
        s_mon = month_names[int(s_date[1])]

        e_date = e_str.split(',')[0].replace('-', '/').split('/')
        e_day = e_date[0]
        e_mon = month_names[int(e_date[1])]

        print(f"{name} : {days} days ({s_day} {s_mon} to {e_day} {e_mon})")

     response   pattern 
Fastest replier : Vikas (avg 34.9 minutes)
Slowest replier : Aman (avg 54.9 minutes)

LONGEST SILENT STREAKS (consecutive days with zero messages)
Vikas : 11 days (22 Apr to 04 May)
Rahul : 0 days (never went silent)
Priya : 0 days (never went silent)
Karan : 0 days (never went silent)
Neha : 0 days (never went silent)
Aman : 0 days (never went silent)


In [ ]:
from datetime import datetime
# to prevent writing seperate 15 archetype we wrote one def to reduce our work load

def assign_archetype(stats, total_group_msgs):
    total = stats['total']
    if total == 0:
        total = 1 # preventing errors

    avg_words = stats['words'] / total

    # calculating score
    scores = {
        "THE NIGHT WATCHMAN": (stats['night'] / total) * 100,
        "THE MORNING ALARM": (stats['morning'] / total) * 100,
        "THE WEEKEND WARRIOR": (stats['weekend'] / total) * 100,
        "THE SPAMMER (MONOLOGUER)": stats['streak'] * 5,
        "THE ESSAYIST": avg_words * 2,
        "THE ONE-WORD WONDER": 50 if avg_words < 2 else 0,
        "THE QUESTION MARK": (stats['questions'] / total) * 100,
        "THE DELETE BUTTON": (stats['deleted'] / total) * 100,
        "THE MEDIA MOGUL": (stats['media'] / total) * 100,
        "THE LINK DROPPER": (stats['links'] / total) * 100,
        "THE LAUGH TRACK": (stats['laughs'] / total) * 100,
        "THE ORGANIZER": (stats['org'] / total) * 100,
        "THE GHOST": 100 if total < (total_group_msgs * 0.05) else 0,
        "THE YAPPER": (total / total_group_msgs) * 100
    }

    # hifhesgg score
    top_archetype = max(scores, key=scores.get)
    top_score = scores[top_archetype]

    return top_archetype, top_score


#main logic loop

user_stats = {}
prev_sender = ""
group_total_msgs = 0

for msg in clean_data:
    sender = msg['name']
    text = msg['message'].lower()
    time_str = msg['time']

    group_total_msgs += 1

    # baseline
    if sender not in user_stats:
        user_stats[sender] = {
            'total': 0, 'night': 0, 'morning': 0, 'weekend': 0,
            'streak': 0, 'current_streak': 0, 'words': 0,
            'questions': 0, 'deleted': 0, 'media': 0, 'links': 0,
            'laughs': 0, 'org': 0
        }

    stats = user_stats[sender]
    stats['total'] += 1

    # 1. words for monolouge speakr
    if sender == prev_sender:
        stats['current_streak'] += 1
        if stats['current_streak'] > stats['streak']:
            stats['streak'] = stats['current_streak']
    else:
        stats['current_streak'] = 1

    # 2. Word for essayist
    words = text.split()
    word_cnt = 0
    for w in words:
        word_cnt += 1
    stats['words'] += word_cnt

    # 3. characters for logic
    for char in text:
        if char == '?':
            stats['questions'] += 1

    if "message was deleted" in text: stats['deleted'] += 1
    if "<media omitted>" in text: stats['media'] += 1
    if "http://" in text or "https://" in text: stats['links'] += 1

    # keywords for laugh
    if text in ['lol', 'lmao', 'haha']:
        stats['laughs'] += 1

    #  words we want to check for organizer
    for w in ['plan', 'meet', 'time', 'where', 'go']:
        if w in text:
            stats['org'] += 1

    # 4. Datetime archetype used
    try:
        dt = datetime.strptime(time_str, '%d/%m/%y, %H:%M')
        if 0 <= dt.hour <= 4: stats['night'] += 1
        if 6 <= dt.hour <= 9: stats['morning'] += 1
        if dt.weekday() >= 4: stats['weekend'] += 1
    except:
        pass

    prev_sender = sender

print("   personality archetype    ")

for person in user_stats:

    archetype_name, final_score = assign_archetype(user_stats[person], group_total_msgs)

    # f-string used for our result
    print(f"{person} → {archetype_name} ({final_score:.1f} score metric)")

   personality archetype    
Rahul → THE SPAMMER (MONOLOGUER) (65.0 score metric)
Priya → THE WEEKEND WARRIOR (40.4 score metric)
Karan → THE ESSAYIST (114.1 score metric)
Neha → THE SPAMMER (MONOLOGUER) (45.0 score metric)
Aman → THE NIGHT WATCHMAN (68.8 score metric)
Vikas → THE GHOST (100.0 score metric)


In [ ]:
dashboard_width = 65

# 1. the title
print("┌" + "=" * (dashboard_width - 2) + "┐")
print("│" + f"{'GroupDNA: HOSTEL CHAT EXECUTIVE SUMMARY':^{dashboard_width - 2}}" + "│")
print("└" + "=" * (dashboard_width - 2) + "┘")

print("│" + f"{' 1. THE RAW NUMBERS':<{dashboard_width - 2}}" + "│")
print("│" + "-" * (dashboard_width - 2) + "│")
print(f"│ Total Messages     : {group_total_msgs:<42}│")
print(f"│ Total Participants : {total_participants:<42}│")
print(f"│ Chat Duration      : {first_msg_date} to {last_msg_date:<28}│")
print("│" + " " * (dashboard_width - 2) + "│")

# 3.absolute peaks
print("│" + f"{' 2. GROUP RHYTHM':<{dashboard_width - 2}}" + "│")
print("│" + "-" * (dashboard_width - 2) + "│")
# hours and days defining them
print(f"│ Busiest Day        : {peak_day:<10} ({peak_day_amount} messages){'':<18}│")
print(f"│ Busiest Hour       : {peak_hour}:00       ({peak_hour_amount} messages){'':<18}│")
print("│" + " " * (dashboard_width - 2) + "│")

# 4. The Archetypes
print("│" + f"{' 3. PERSONALITY ARCHETYPES':<{dashboard_width - 2}}" + "│")
print("│" + "-" * (dashboard_width - 2) + "│")

for person in user_stats:
    # single scoring
    arch_name, score = assign_archetype(user_stats[person], group_total_msgs)

    print(f"│ {person:<10} → {arch_name:<35} ({score:>4.1f}) │")

print("└" + "=" * (dashboard_width - 2) + "┘")

┌===============================================================┐
│            GroupDNA: HOSTEL CHAT EXECUTIVE SUMMARY            │
└===============================================================┘
│ 1. THE RAW NUMBERS                                            │
│---------------------------------------------------------------│
│ Total Messages     : 3127                                      │
│ Total Participants : 6                                         │
│ Chat Duration      : 01/04/24 to 30/05/24                    │
│                                                               │
│ 2. GROUP RHYTHM                                               │
│---------------------------------------------------------------│
│ Busiest Day        : 04/05/24   (74 messages)                  │
│ Busiest Hour       : 18:00       (244 messages)                  │
│                                                               │
│ 3. PERSONALITY ARCHETYPES                                     │
│----